# **Feature Engineering - UK Land Registry Price Prediction**

This notebook performs feature engineering on the property price data.

**Objectives:**
- Load Parquet data
- Create temporal features
- Engineer location-based features
- Encode categorical variables
- Create aggregated features
- Handle missing values
- Feature scaling and normalization

In [1]:
import os
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, year, month, dayofweek, quarter, datediff, lit, to_date,
    when, concat, substring, length, avg, count, stddev,
    lag, lead, row_number, rank, dense_rank
)
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## **1. Initialize Spark Session**

In [2]:
spark = SparkSession.builder \
    .appName("Feature_Engineering") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

log4j = spark.sparkContext._jvm.org.apache.log4j
log4j.LogManager.getLogger("org.apache.spark.storage.MemoryStore").setLevel(log4j.Level.ERROR)
log4j.LogManager.getLogger("org.apache.spark.storage.BlockManager").setLevel(log4j.Level.ERROR)
log4j.LogManager.getLogger("org.apache.spark.scheduler.DAGScheduler").setLevel(log4j.Level.ERROR)
print(f"Spark Version: {spark.version}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 12:49:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/27 12:49:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Version: 3.5.0


## **2. Load Parquet Data**

In [3]:
df = spark.read.parquet("../data/property_prices.parquet")

print(f"Data loaded: {df.count():,} records")
print(f"Partitions: {df.rdd.getNumPartitions()}")
df.printSchema()

Data loaded: 6,721,312 records
Partitions: 5
root
 |-- transaction_id: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- date_of_transfer: date (nullable = true)
 |-- postcode: string (nullable = true)
 |-- old_new: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- paon: string (nullable = true)
 |-- saon: string (nullable = true)
 |-- street: string (nullable = true)
 |-- locality: string (nullable = true)
 |-- town_city: string (nullable = true)
 |-- district: string (nullable = true)
 |-- county: string (nullable = true)
 |-- ppd_category: string (nullable = true)
 |-- record_status: string (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- property_type: string (nullable = true)



In [4]:
df.cache()
print(f"Data cached in memory")
print(f"Storage Level: {df.storageLevel}")

Data cached in memory
Storage Level: Disk Memory Deserialized 1x Replicated


## **3. Temporal Features**

In [5]:
df_temporal = df \
    .withColumn("quarter", quarter("date_of_transfer")) \
    .withColumn("day_of_week", dayofweek("date_of_transfer")) \
    .withColumn("days_since_2019", datediff(col("date_of_transfer"), to_date(lit("2019-01-01")))) \
    .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), 1).otherwise(0))

print("Temporal features created")
df_temporal.select("date_of_transfer", "year", "month", "quarter", "day_of_week", "is_weekend").show(5)

Temporal features created


+----------------+----+-----+-------+-----------+----------+
|date_of_transfer|year|month|quarter|day_of_week|is_weekend|
+----------------+----+-----+-------+-----------+----------+
|      2021-08-06|2021|    8|      3|          6|         0|
|      2021-09-10|2021|    9|      3|          6|         0|
|      2021-08-27|2021|    8|      3|          6|         0|
|      2021-06-25|2021|    6|      2|          6|         0|
|      2021-08-26|2021|    8|      3|          5|         0|
+----------------+----+-----+-------+-----------+----------+
only showing top 5 rows



## **4. Location-Based Features**

In [6]:
df_location = df_temporal \
    .withColumn("postcode_area", substring("postcode", 1, 2)) \
    .withColumn("postcode_district", substring("postcode", 1, 4)) \
    .withColumn("has_locality", when(col("locality").isNotNull(), 1).otherwise(0)) \
    .withColumn("has_street", when(col("street").isNotNull(), 1).otherwise(0))

print("Location features created")
df_location.select("postcode", "postcode_area", "postcode_district", "has_locality").show(5)

Location features created
+--------+-------------+-----------------+------------+
|postcode|postcode_area|postcode_district|has_locality|
+--------+-------------+-----------------+------------+
|SO45 2HT|           SO|             SO45|           1|
|SP11 6TU|           SP|             SP11|           0|
|SO51 0AX|           SO|             SO51|           0|
|PO12 4NF|           PO|             PO12|           1|
|SO15 8PW|           SO|             SO15|           0|
+--------+-------------+-----------------+------------+
only showing top 5 rows



## **5. Aggregated Features**

In [7]:
area_stats = df_location.groupBy("postcode_area").agg(
    avg("price").alias("area_avg_price"),
    count("*").alias("area_transaction_count"),
    stddev("price").alias("area_price_stddev")
)

df_area_features = df_location.join(area_stats, on="postcode_area", how="left")

print("Area-level aggregated features created")
df_area_features.select("postcode_area", "price", "area_avg_price", "area_transaction_count").show(5)

Area-level aggregated features created


+-------------+------+------------------+----------------------+
|postcode_area| price|    area_avg_price|area_transaction_count|
+-------------+------+------------------+----------------------+
|           RG|300000| 504246.5194837929|                 99185|
|           SO|260000| 431264.4696243108|                 77990|
|           SO|360000| 431264.4696243108|                 77990|
|           SO|260250| 431264.4696243108|                 77990|
|           SP|295000|390249.99236564874|                 29996|
+-------------+------+------------------+----------------------+
only showing top 5 rows



In [8]:
district_stats = df_location.groupBy("district").agg(
    avg("price").alias("district_avg_price"),
    count("*").alias("district_transaction_count")
)

df_district_features = df_area_features.join(district_stats, on="district", how="left")

print("District-level aggregated features created")

District-level aggregated features created


In [9]:
property_type_stats = df_location.groupBy("property_type", "year").agg(
    avg("price").alias("property_year_avg_price")
)

df_prop_features = df_district_features.join(
    property_type_stats, 
    on=["property_type", "year"], 
    how="left"
)

print("Property type yearly features created")

Property type yearly features created


## **6. Derived Features**

In [10]:
df_derived = df_prop_features \
    .withColumn("price_vs_area_avg", 
                (col("price") - col("area_avg_price")) / col("area_avg_price")) \
    .withColumn("price_vs_district_avg", 
                (col("price") - col("district_avg_price")) / col("district_avg_price")) \
    .withColumn("price_vs_property_avg", 
                (col("price") - col("property_year_avg_price")) / col("property_year_avg_price"))

print("Derived features created")
df_derived.select("price", "price_vs_area_avg", "price_vs_district_avg").show(5)

Derived features created


+------+--------------------+---------------------+
| price|   price_vs_area_avg|price_vs_district_avg|
+------+--------------------+---------------------+
|300000| -0.4050529088289674| -0.31159872360680524|
|260000|-0.39712167750221833| -0.45591995185167783|
|360000|-0.16524539961845616|  -0.2076833675857858|
|260250|-0.39654198680750896| -0.12403823729973605|
|295000| -0.2440742965509229| -0.35074053732724114|
+------+--------------------+---------------------+
only showing top 5 rows



## **7. Handle Missing Values**

In [11]:
null_before = df_derived.select([count(when(col(c).isNull(), c)).alias(c) for c in df_derived.columns])
print("Null counts before imputation:")
null_before.show()

Null counts before imputation:


26/02/27 12:49:30 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+----+--------+-------------+--------------+-----+----------------+--------+-------+--------+----+-------+------+--------+---------+------+------------+-------------+-----+-------+-----------+---------------+----------+-----------------+------------+----------+--------------+----------------------+-----------------+------------------+--------------------------+-----------------------+-----------------+---------------------+---------------------+
|property_type|year|district|postcode_area|transaction_id|price|date_of_transfer|postcode|old_new|duration|paon|   saon|street|locality|town_city|county|ppd_category|record_status|month|quarter|day_of_week|days_since_2019|is_weekend|postcode_district|has_locality|has_street|area_avg_price|area_transaction_count|area_price_stddev|district_avg_price|district_transaction_count|property_year_avg_price|price_vs_area_avg|price_vs_district_avg|price_vs_property_avg|
+-------------+----+--------+-------------+--------------+-----+--------

In [12]:
df_imputed = df_derived.fillna({
    "area_avg_price": 0,
    "area_price_stddev": 0,
    "area_transaction_count": 0,
    "district_avg_price": 0,
    "district_transaction_count": 0,
    "property_year_avg_price": 0,
    "price_vs_area_avg": 0,
    "price_vs_district_avg": 0,
    "price_vs_property_avg": 0,
    "locality": "UNKNOWN",
    "street": "UNKNOWN",
    "county": "UNKNOWN"
})

print("Missing values imputed")

Missing values imputed


## **8. Encode Categorical Variables**

In [13]:
categorical_cols = ["property_type", "old_new", "duration", "postcode_area", "county"]

indexers = [StringIndexer(inputCol=col, outputCol=col+"_index", handleInvalid="keep") 
            for col in categorical_cols]

encoders = [OneHotEncoder(inputCol=col+"_index", outputCol=col+"_encoded") 
            for col in categorical_cols]

print(f"Created {len(indexers)} indexers and {len(encoders)} encoders")

Created 5 indexers and 5 encoders


## **9. Feature Vector Assembly**

In [14]:
numeric_features = [
    "year", "month", "quarter", "day_of_week", "is_weekend", "days_since_2019",
    "has_locality", "has_street",
    "area_avg_price", "area_transaction_count", "area_price_stddev",
    "district_avg_price", "district_transaction_count",
    "property_year_avg_price",
    "price_vs_area_avg", "price_vs_district_avg", "price_vs_property_avg"
]

encoded_features = [col+"_encoded" for col in categorical_cols]

all_features = numeric_features + encoded_features

assembler = VectorAssembler(
    inputCols=all_features,
    outputCol="features_unscaled",
    handleInvalid="skip"
)

print(f"Assembler will combine {len(all_features)} feature columns")

Assembler will combine 22 feature columns


## **10. Feature Scaling**

In [15]:
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withMean=False,
    withStd=True
)

print("Scaler configured")

Scaler configured


## **11. Build Feature Engineering Pipeline**

In [16]:
pipeline_stages = indexers + encoders + [assembler, scaler]

feature_pipeline = Pipeline(stages=pipeline_stages)

print(f"Pipeline created with {len(pipeline_stages)} stages")

Pipeline created with 12 stages


## **12. Fit and Transform**

In [17]:
start_time = time.time()

pipeline_model = feature_pipeline.fit(df_imputed)

fit_time = time.time() - start_time
print(f"Pipeline fitted in {fit_time:.2f} seconds")

Pipeline fitted in 27.69 seconds


In [18]:
start_time = time.time()

df_features = pipeline_model.transform(df_imputed)

transform_time = time.time() - start_time
print(f"Pipeline transformed in {transform_time:.2f} seconds")

Pipeline transformed in 0.28 seconds


In [19]:
df_features = df_features.withColumn("label", col("price"))

print("Label column created")
df_features.select("price", "label", "features").show(5, truncate=False)

Label column created


+------+------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|price |label |features                                                                                                                                                                                                                                                                                                                                                                                                                                                                   |
+------+------+---------------------------------

## **13. Filter Invalid Records**

In [20]:
df_final = df_features.filter(
    (col("price").isNotNull()) & 
    (col("price") > 0) & 
    (col("price") < 10000000) &
    (col("features").isNotNull())
)

print(f"Records before filtering: {df_features.count():,}")
print(f"Records after filtering: {df_final.count():,}")

Records before filtering: 6,721,312


Records after filtering: 6,713,112


## **14. Persist Features**

In [21]:
df_final.persist()
print(f"Final dataset persisted with {df_final.count():,} records")

26/02/27 12:51:40 WARN MemoryStore: Not enough space to cache rdd_327_0 in memory! (computed 993.9 MiB so far)
26/02/27 12:51:45 WARN MemoryStore: Not enough space to cache rdd_327_0 in memory! (computed 644.9 MiB so far)
26/02/27 12:51:48 WARN MemoryStore: Not enough space to cache rdd_327_4 in memory! (computed 410.6 MiB so far)
26/02/27 12:51:51 WARN MemoryStore: Not enough space to cache rdd_327_6 in memory! (computed 409.7 MiB so far)
26/02/27 12:51:56 WARN MemoryStore: Not enough space to cache rdd_327_5 in memory! (computed 643.8 MiB so far)
26/02/27 12:52:06 WARN MemoryStore: Not enough space to cache rdd_327_6 in memory! (computed 409.7 MiB so far)
26/02/27 12:52:06 WARN MemoryStore: Not enough space to cache rdd_327_5 in memory! (computed 643.8 MiB so far)
26/02/27 12:52:07 WARN MemoryStore: Not enough space to cache rdd_327_9 in memory! (computed 62.4 MiB so far)
26/02/27 12:52:07 WARN MemoryStore: Not enough space to cache rdd_327_9 in memory! (computed 62.4 MiB so far)
26/

Final dataset persisted with 6,713,112 records


## **15. Save Processed Data**

In [22]:
output_path = "../data/processed_features.parquet"

df_final.write \
    .mode("overwrite") \
    .parquet(output_path)

print(f"Processed features saved to {output_path}")

26/02/27 12:52:10 WARN MemoryStore: Not enough space to cache rdd_327_0 in memory! (computed 411.7 MiB so far)
26/02/27 12:52:36 WARN MemoryStore: Not enough space to cache rdd_327_4 in memory! (computed 410.6 MiB so far)
26/02/27 12:52:38 WARN MemoryStore: Not enough space to cache rdd_327_5 in memory! (computed 236.9 MiB so far)
26/02/27 12:52:39 WARN MemoryStore: Not enough space to cache rdd_327_6 in memory! (computed 61.4 MiB so far)


Processed features saved to ../data/processed_features.parquet


In [23]:
pipeline_model.write().overwrite().save("../data/feature_pipeline_model")
print("Feature pipeline model saved")

Feature pipeline model saved


## **16. Feature Statistics**

In [24]:
print("\n=== Feature Engineering Summary ===")
print(f"Total Features Created: {len(all_features)}")
print(f"Numeric Features: {len(numeric_features)}")
print(f"Encoded Features: {len(encoded_features)}")
print(f"Final Records: {df_final.count():,}")
print(f"Pipeline Fit Time: {fit_time:.2f} seconds")
print(f"Pipeline Transform Time: {transform_time:.2f} seconds")


=== Feature Engineering Summary ===
Total Features Created: 22
Numeric Features: 17
Encoded Features: 5


26/02/27 12:53:07 WARN MemoryStore: Not enough space to cache rdd_327_6 in memory! (computed 119.4 MiB so far)
26/02/27 12:53:07 WARN MemoryStore: Not enough space to cache rdd_327_5 in memory! (computed 120.3 MiB so far)
26/02/27 12:53:07 WARN MemoryStore: Not enough space to cache rdd_327_4 in memory! (computed 237.0 MiB so far)
26/02/27 12:53:07 WARN MemoryStore: Not enough space to cache rdd_327_0 in memory! (computed 411.7 MiB so far)


Final Records: 6,713,112
Pipeline Fit Time: 27.69 seconds
Pipeline Transform Time: 0.28 seconds


In [25]:
sample_features = df_final.select("price", "label", "features").limit(1000).toPandas()
sample_features.to_csv("../data/samples/features_sample.csv", index=False)
print("Feature sample saved")

Feature sample saved


26/02/27 12:53:09 WARN MemoryStore: Not enough space to cache rdd_327_0 in memory! (computed 411.7 MiB so far)
